<a href="https://colab.research.google.com/github/nicolasengland14-gif/ECON5200-Applied-Data-Analytics-in-Economics/blob/main/Midterm%20Project/notebooks/02_Replication_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv(
    "data/raw/public.dat",
    header=None,
    sep=r"\s+",
)

df_raw.head()

,0,1,2,3,4,5,6,7,8,9,...,36,37,38,39,40,41,42,43,44,45
0,46,1,0,0,0,0,0,1,0,0,...,0.08,1,2,6.50,16.50,1.03,.,0.94,4,4
1,49,2,0,0,0,0,0,1,0,0,...,0.05,0,2,10.00,13.00,1.01,0.89,2.35,4,4
2,506,2,1,0,0,0,0,1,0,0,...,0.25,.,1,11.00,11.00,0.95,0.74,2.33,4,3
3,56,4,1,0,0,0,0,1,0,0,...,0.15,0,2,10.00,12.00,0.92,0.79,0.87,2,2
4,61,4,1,0,0,0,0,1,0,0,...,0.15,0,2,10.00,12.00,1.01,0.84,0.95,2,2


In [12]:
df_raw = df_raw.replace(".", pd.NA)
df_raw = df_raw.apply(pd.to_numeric)
df_raw[[31,32,33,39,40,44]].describe()

,31,32,33,39,40,44
count,398.000000,400.000000,404.000000,399.000000,399.000000,388.000000
mean,8.275126,18.677500,3.483911,8.110902,14.465539,3.608247
std,7.970763,10.699635,1.139898,2.157961,2.752495,1.243540
min,0.000000,0.000000,0.000000,0.000000,8.000000,1.000000
25%,2.000000,11.000000,3.000000,7.000000,12.000000,3.000000
50%,6.000000,17.000000,3.000000,7.000000,15.000000,3.000000
75%,12.000000,25.000000,4.000000,10.500000,16.000000,4.000000
max,40.000000,60.000000,8.000000,11.500000,24.000000,8.000000


In [13]:
df_raw["fulltime_wave1"] = df_raw[33] + df_raw[34] + 0.5 * df_raw[32]

df_raw["fulltime_wave2"] = df_raw[40] + df_raw[44] + 0.5 * df_raw[39]

In [14]:
df_raw["treat"] = (df_raw[2] == 1).astype(int)

nj_before = df_raw[df_raw["treat"]==1]["fulltime_wave1"].mean()
nj_after  = df_raw[df_raw["treat"]==1]["fulltime_wave2"].mean()

pa_before = df_raw[df_raw["treat"]==0]["fulltime_wave1"].mean()
pa_after  = df_raw[df_raw["treat"]==0]["fulltime_wave2"].mean()

did = (nj_after - nj_before) - (pa_after - pa_before)

print(did)

2.322326964748843


In [15]:
df_employment = pd.DataFrame({
    "store": df_raw[0],
    "treat": df_raw["treat"],
    "employment_before": df_raw["fulltime_wave1"],
    "employment_after": df_raw["fulltime_wave2"]
})

df_employment.head()

,store,treat,employment_before,employment_after
0,46,0,24.80,23.75
1,49,0,15.95,22.00
2,506,1,12.50,20.50
3,56,1,25.25,19.00
4,61,1,12.25,19.00


In [16]:
df_raw["wage_wave1"] = df_raw[34]
df_raw["wage_wave2"] = df_raw[41]

df_raw["price_wave1"] = df_raw[43]
df_raw["price_wave2"] = df_raw[44]
table2 = df_raw.groupby("treat")[[
    "fulltime_wave1",
    "fulltime_wave2",
    "wage_wave1",
    "wage_wave2",
    "price_wave1",
    "price_wave2"
]].agg(["mean","std"])

table2.index = ["PA","NJ"]

table2

fulltime_wave1           fulltime_wave2           wage_wave1            \
             mean       std           mean       std       mean       std   
PA      18.658889  5.680742      21.969595  2.238473   4.983619  0.279604   
NJ      16.822045  5.200483      22.455078  2.516370   5.020909  0.190177   

   wage_wave2           price_wave1           price_wave2            
         mean       std        mean       std        mean       std  
PA   1.045349  0.093564    1.305765  0.644682    3.357692  1.162036  
NJ   1.044154  0.093930    1.448092  0.651591    4.117188  1.252532

In [17]:
df_employment_long = df_employment.melt(
    id_vars=["store","treat"],
    value_vars=["employment_before","employment_after"],
    var_name="period",
    value_name="employment"
)

df_employment_long["post"] = (
    df_employment_long["period"] == "employment_after"
).astype(int)

In [18]:
import statsmodels.formula.api as smf

df_regression = df_employment_long.dropna(
    subset=["employment","treat","post","store"]
).copy()

model = smf.ols("employment ~ treat * post", data=df_regression)

results = model.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_regression["store"]}
)

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             employment   R-squared:                       0.206
Model:                            OLS   Adj. R-squared:                  0.203
Method:                 Least Squares   F-statistic:                     90.82
Date:                Tue, 10 Mar 2026   Prob (F-statistic):           9.67e-45
Time:                        04:38:46   Log-Likelihood:                -2204.4
No. Observations:                 771   AIC:                             4417.
Df Residuals:                     767   BIC:                             4435.
Df Model:                           3                                         
Covariance Type:              cluster                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     18.6589      0.358     52.077      0.0